![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [ ]:
# -*- coding: utf-8 -*-
"""
Research Pipeline v12.0
FOCO: único candidato rescatable
- Ticker: QQQ
- Direction: Put
- Regime: bear_volatile
- Label: 7d / TP 2.0 ATR / SL 1.25 ATR
- Modelo: LightGBM no calibrado

Objetivo:
- micro-tuning local del mejor candidato
- seleccionar por discriminación + ranking + utilidad
- promover solo a "paper_candidate", no producción final
"""

import warnings
warnings.filterwarnings("ignore")

import json
import pickle
from datetime import datetime
from itertools import product

import numpy as np
import pandas as pd

from QuantConnect.Research import QuantBook
from QuantConnect import Resolution

try:
    import pandas_ta as ta
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import TimeSeriesSplit
    from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
    from lightgbm import LGBMClassifier
    HAS_DEPS = True
except ImportError as e:
    HAS_DEPS = False
    print(f"Faltan dependencias críticas: {e}")


RANDOM_STATE = 42
START_DATE = datetime(2015, 8, 1)
END_DATE = datetime(2025, 8, 14)

TICKER = "QQQ"
SUPPORT = ["VIXY"]

LABEL_CFG = {
    "label_name": "B_7d_2p0_1p25",
    "look_forward_days": 7,
    "tp_atr_mult": 2.0,
    "sl_atr_mult": 1.25
}

CV_SPLITS = 4
MIN_SAMPLES = 350
THRESHOLD_GRID = np.linspace(0.45, 0.90, 91)

# Gates paper-candidate
PAPER_GATES = {
    "min_samples": 350,
    "min_auc": 0.63,
    "min_lift10": 1.20,
    "min_pr_auc_over_base": 0.03,
    "max_brier": 0.24,
    "min_auc_stability": 0.50,
    "min_signal_precision": 0.60,
    "min_utility": 0.15
}

# Grid local pequeño pero útil
PARAM_GRID = {
    "n_estimators": [150, 250, 400],
    "learning_rate": [0.03, 0.05],
    "num_leaves": [15, 31, 63],
    "min_child_samples": [15, 25, 40],
    "subsample": [0.85, 0.95],
    "colsample_bytree": [0.85, 0.95],
    "reg_alpha": [0.0, 0.2],
    "reg_lambda": [0.2, 0.6]
}

MAX_COMBINATIONS = 48  # limitar coste


def safe_auc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_prob))


def safe_pr_auc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(average_precision_score(y_true, y_prob))


def expected_calibration_error(y_true, y_prob, n_bins=10):
    if len(y_true) == 0:
        return np.nan

    df = pd.DataFrame({"y_true": y_true, "y_prob": y_prob}).sort_values("y_prob")
    try:
        df["bin"] = pd.qcut(df["y_prob"], q=n_bins, labels=False, duplicates="drop")
    except Exception:
        return np.nan

    ece = 0.0
    uniq = df["bin"].nunique()
    if uniq == 0:
        return np.nan

    for i in range(uniq):
        part = df[df["bin"] == i]
        if len(part) == 0:
            continue
        acc = float(part["y_true"].mean())
        conf = float(part["y_prob"].mean())
        ece += (len(part) / len(df)) * abs(acc - conf)

    return float(ece)


def lift_at_k(y_true, y_prob, k=0.10):
    n = len(y_true)
    if n == 0:
        return np.nan, np.nan, np.nan

    k_n = max(1, int(np.floor(n * k)))
    order = np.argsort(-y_prob)
    idx = order[:k_n]

    precision_k = float(y_true[idx].mean()) if k_n > 0 else np.nan
    base_rate = float(np.mean(y_true)) if n > 0 else np.nan
    lift = float(precision_k / base_rate) if base_rate > 0 else np.nan
    return lift, precision_k, base_rate


def utility_score_from_probs(y_true, y_prob, threshold, tp_reward, sl_cost):
    signal = (y_prob >= threshold).astype(int)
    n_signals = int(signal.sum())

    if n_signals == 0:
        return -999.0, 0, 0.0

    selected = y_true[signal == 1]
    precision = float(selected.mean()) if len(selected) > 0 else 0.0
    utility = precision * tp_reward - (1.0 - precision) * sl_cost
    return float(utility), n_signals, precision


def best_threshold_by_utility(y_true, y_prob, tp_reward, sl_cost, min_signal_rate=0.02, max_signal_rate=0.20):
    best_thr = 0.70
    best_score = -999.0

    n = len(y_true)
    if n == 0:
        return best_thr

    for thr in THRESHOLD_GRID:
        signal = (y_prob >= thr).astype(int)
        signal_rate = float(signal.sum()) / n

        if signal.sum() == 0:
            continue
        if signal_rate < min_signal_rate or signal_rate > max_signal_rate:
            continue

        utility, _, _ = utility_score_from_probs(
            y_true=y_true,
            y_prob=y_prob,
            threshold=thr,
            tp_reward=tp_reward,
            sl_cost=sl_cost
        )

        if utility > best_score:
            best_score = utility
            best_thr = thr

    return float(best_thr)


def detect_regimes(df, support_hist):
    regimes = pd.DataFrame(index=df.index)

    c = df["close"]
    sma200 = ta.sma(c, 200)

    vix_df = support_hist.get("VIXY")
    if vix_df is None or sma200 is None:
        return pd.DataFrame(index=df.index, columns=["bear_volatile"]).fillna(False)

    vix = vix_df["close"].reindex(c.index, method="ffill")

    is_bear = c <= sma200
    is_volatile = vix > 22

    regimes["bear_volatile"] = is_bear & is_volatile
    return regimes.fillna(False)


def get_triple_barrier_labels(hist_df, direction, config):
    c = hist_df["close"]
    h = hist_df["high"]
    l = hist_df["low"]

    labels = pd.Series(np.nan, index=c.index)
    atr = ta.atr(h, l, c, length=14)

    if atr is None:
        return labels.fillna(0).astype(int)

    tp_mult = config["tp_atr_mult"]
    sl_mult = config["sl_atr_mult"]
    max_days = config["look_forward_days"]

    n = len(c)
    for i in range(n - max_days):
        entry = c.iloc[i]
        atr_val = atr.iloc[i]

        if pd.isna(atr_val) or atr_val == 0:
            continue

        if direction == "Put":
            tp_level = entry - atr_val * tp_mult
            sl_level = entry + atr_val * sl_mult
        else:
            tp_level = entry + atr_val * tp_mult
            sl_level = entry - atr_val * sl_mult

        decided = False
        for j in range(1, max_days + 1):
            if direction == "Put":
                if l.iloc[i + j] <= tp_level:
                    labels.iloc[i] = 1
                    decided = True
                    break
                if h.iloc[i + j] >= sl_level:
                    labels.iloc[i] = 0
                    decided = True
                    break
            else:
                if h.iloc[i + j] >= tp_level:
                    labels.iloc[i] = 1
                    decided = True
                    break
                if l.iloc[i + j] <= sl_level:
                    labels.iloc[i] = 0
                    decided = True
                    break

        if not decided:
            labels.iloc[i] = 0

    return labels.dropna().astype(int)


def compute_features(df, support_hist, yield_curve):
    c = df["close"]
    h = df["high"]
    l = df["low"]
    o = df["open"]
    v = df.get("volume")

    feats = pd.DataFrame(index=df.index)

    feats["ret1"] = c.pct_change(1)
    feats["ret3"] = c.pct_change(3)
    feats["ret5"] = c.pct_change(5)
    feats["ret10"] = c.pct_change(10)
    feats["ret21"] = c.pct_change(21)

    sma10 = ta.sma(c, 10)
    sma20 = ta.sma(c, 20)
    sma50 = ta.sma(c, 50)
    sma200 = ta.sma(c, 200)
    ema8 = ta.ema(c, 8)
    ema21 = ta.ema(c, 21)

    if sma10 is not None and sma50 is not None:
        feats["sma10_sma50_ratio"] = sma10 / sma50 - 1.0
    if sma20 is not None and sma200 is not None:
        feats["sma20_sma200_ratio"] = sma20 / sma200 - 1.0
    if sma50 is not None and sma200 is not None:
        feats["sma50_sma200_ratio"] = sma50 / sma200 - 1.0
    if ema8 is not None and ema21 is not None:
        feats["ema8_ema21_ratio"] = ema8 / ema21 - 1.0

    feats["rsi5"] = ta.rsi(c, 5)
    feats["rsi14"] = ta.rsi(c, 14)

    atr14 = ta.atr(h, l, c, length=14)
    feats["atr14_pct"] = atr14 / c

    feats["realized_vol_5"] = feats["ret1"].rolling(5).std()
    feats["realized_vol_10"] = feats["ret1"].rolling(10).std()
    feats["realized_vol_21"] = feats["ret1"].rolling(21).std()
    feats["vol_ratio_5_21"] = feats["realized_vol_5"] / (feats["realized_vol_21"] + 1e-9)

    roll_max_20 = c.rolling(20).max()
    roll_min_20 = c.rolling(20).min()
    feats["dist_to_20d_high"] = c / roll_max_20 - 1.0
    feats["dist_to_20d_low"] = c / roll_min_20 - 1.0

    roll_max_63 = c.rolling(63).max()
    roll_min_63 = c.rolling(63).min()
    feats["dist_to_63d_high"] = c / roll_max_63 - 1.0
    feats["dist_to_63d_low"] = c / roll_min_63 - 1.0

    bb = ta.bbands(c, length=20, std=2.0)
    if bb is not None and "BBU_20_2.0" in bb.columns and "BBL_20_2.0" in bb.columns:
        feats["bb_width"] = (bb["BBU_20_2.0"] - bb["BBL_20_2.0"]) / c
        feats["bb_pos"] = (c - bb["BBL_20_2.0"]) / ((bb["BBU_20_2.0"] - bb["BBL_20_2.0"]) + 1e-9)

    macd = ta.macd(c)
    if macd is not None:
        for col in macd.columns:
            feats[col.lower()] = macd[col]

    adx = ta.adx(h, l, c, length=14)
    if adx is not None:
        for col in adx.columns:
            feats[col.lower()] = adx[col]

    if v is not None and not v.isnull().all():
        obv = ta.obv(c, v)
        if obv is not None:
            feats["obv_slope5"] = obv.pct_change(5)
            feats["obv_slope10"] = obv.pct_change(10)
        feats["vol_z20"] = (v - v.rolling(20).mean()) / (v.rolling(20).std() + 1e-9)
        feats["vol_ratio_5_20"] = v.rolling(5).mean() / (v.rolling(20).mean() + 1e-9)

    feats["gap"] = o / c.shift(1) - 1.0
    feats["range_pct"] = (h - l) / c
    feats["close_to_high"] = (c - l) / ((h - l) + 1e-9)
    feats["close_to_low"] = (h - c) / ((h - l) + 1e-9)

    vix_df = support_hist.get("VIXY")
    if vix_df is not None:
        vix_close = vix_df["close"].reindex(c.index, method="ffill")
        feats["vix_level"] = vix_close
        feats["vix_ret3"] = vix_close.pct_change(3)
        feats["vix_ret5"] = vix_close.pct_change(5)
        feats["vix_rsi14"] = ta.rsi(vix_close, 14)
        feats["vix_vs_asset_ret5"] = vix_close.pct_change(5) - c.pct_change(5)

    if yield_curve is not None and not yield_curve.empty:
        feats["yield_curve_slope"] = yield_curve.reindex(c.index.normalize(), method="ffill").values

    feats = feats.replace([np.inf, -np.inf], np.nan)
    feats = feats.bfill().ffill().dropna()
    return feats


def build_lgbm(params):
    return LGBMClassifier(
        random_state=RANDOM_STATE,
        objective="binary",
        n_estimators=int(params["n_estimators"]),
        learning_rate=float(params["learning_rate"]),
        num_leaves=int(params["num_leaves"]),
        min_child_samples=int(params["min_child_samples"]),
        subsample=float(params["subsample"]),
        colsample_bytree=float(params["colsample_bytree"]),
        reg_alpha=float(params["reg_alpha"]),
        reg_lambda=float(params["reg_lambda"])
    )


def generate_param_combinations():
    keys = list(PARAM_GRID.keys())
    vals = [PARAM_GRID[k] for k in keys]
    combos = []
    for combo in product(*vals):
        d = dict(zip(keys, combo))
        if d["num_leaves"] == 63 and d["min_child_samples"] == 40:
            continue
        combos.append(d)

    # recorte determinista
    combos = combos[:MAX_COMBINATIONS]
    return combos


def paper_candidate_decision(row):
    reasons = []

    if row["Samples"] < PAPER_GATES["min_samples"]:
        reasons.append(f"samples<{PAPER_GATES['min_samples']}")
    if pd.isna(row["AUC (Mean)"]) or row["AUC (Mean)"] < PAPER_GATES["min_auc"]:
        reasons.append(f"auc<{PAPER_GATES['min_auc']}")
    if pd.isna(row["Lift@10% (Mean)"]) or row["Lift@10% (Mean)"] < PAPER_GATES["min_lift10"]:
        reasons.append(f"lift10<{PAPER_GATES['min_lift10']}")
    if pd.isna(row["PR_AUC (Mean)"]) or (row["PR_AUC (Mean)"] - row["BaseRate"]) < PAPER_GATES["min_pr_auc_over_base"]:
        reasons.append(f"pr_auc_minus_base<{PAPER_GATES['min_pr_auc_over_base']}")
    if pd.isna(row["Brier (Mean)"]) or row["Brier (Mean)"] > PAPER_GATES["max_brier"]:
        reasons.append(f"brier>{PAPER_GATES['max_brier']}")
    if pd.isna(row["AUC Stability"]) or row["AUC Stability"] < PAPER_GATES["min_auc_stability"]:
        reasons.append(f"auc_stability<{PAPER_GATES['min_auc_stability']}")
    if pd.isna(row["SignalPrecision (Mean)"]) or row["SignalPrecision (Mean)"] < PAPER_GATES["min_signal_precision"]:
        reasons.append(f"signal_precision<{PAPER_GATES['min_signal_precision']}")
    if pd.isna(row["Utility (Mean)"]) or row["Utility (Mean)"] < PAPER_GATES["min_utility"]:
        reasons.append(f"utility<{PAPER_GATES['min_utility']}")

    return len(reasons) == 0, ";".join(reasons)


class FocusedResearchV12:
    def __init__(self):
        self.qb = QuantBook()
        self.histories = {}
        self.yield_curve = pd.Series(dtype=float)
        self.results = []

    def load_data(self):
        tickers = [TICKER] + SUPPORT
        symbols = {t: self.qb.AddEquity(t, Resolution.Daily).Symbol for t in tickers}
        raw = self.qb.History(list(symbols.values()), START_DATE, END_DATE, Resolution.Daily)

        for t, s in symbols.items():
            if s in raw.index:
                df = raw.loc[s].copy()
                df.columns = [c.lower() for c in df.columns]
                self.histories[t] = df

        try:
            from QuantConnect.Data.Custom import Quandl
            t10y_symbol = self.qb.AddData(Quandl, "USTREASURY/YIELD.10", Resolution.Daily).Symbol
            t2y_symbol = self.qb.AddData(Quandl, "USTREASURY/YIELD.2", Resolution.Daily).Symbol
            yc_raw = self.qb.History([t10y_symbol, t2y_symbol], START_DATE, END_DATE, Resolution.Daily)

            if t10y_symbol in yc_raw.index and t2y_symbol in yc_raw.index:
                t10y = yc_raw.loc[t10y_symbol]["value"]
                t2y = yc_raw.loc[t2y_symbol]["value"]
                self.yield_curve = (t10y - t2y).reindex(
                    pd.date_range(START_DATE, END_DATE, freq="B"),
                    method="ffill"
                )
        except Exception as e:
            print(f"Advertencia curva de tipos: {e}")

    def run(self):
        self.load_data()

        qqq = self.histories.get(TICKER)
        vixy = self.histories.get("VIXY")

        if qqq is None or vixy is None:
            print("Faltan datos de QQQ o VIXY.")
            return

        support_hist = {"VIXY": vixy}
        regimes = detect_regimes(qqq, support_hist)
        features = compute_features(qqq, support_hist, self.yield_curve)
        labels = get_triple_barrier_labels(qqq, "Put", LABEL_CFG)

        X_all, y_all = features.align(labels, join="inner", axis=0)
        mask = regimes["bear_volatile"].reindex(X_all.index, fill_value=False)

        X = X_all[mask]
        y = y_all[mask]

        if len(X) < MIN_SAMPLES or y.nunique() < 2:
            print(f"Muestras insuficientes tras filtro de régimen: {len(X)}")
            return

        print(f"Muestras finales: {len(X)} | BaseRate={y.mean():.4f}")

        combos = generate_param_combinations()
        print(f"Combinaciones a evaluar: {len(combos)}")

        tscv = TimeSeriesSplit(n_splits=CV_SPLITS)

        for i, params in enumerate(combos, start=1):
            fold_rows = []
            fold_thresholds = []

            for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
                X_train = X.iloc[train_idx]
                X_test = X.iloc[test_idx]
                y_train = y.iloc[train_idx]
                y_test = y.iloc[test_idx]

                if len(X_train) < 120 or y_train.nunique() < 2:
                    continue

                scaler = StandardScaler().fit(X_train)
                X_train_s = scaler.transform(X_train)
                X_test_s = scaler.transform(X_test)

                model = build_lgbm(params)
                model.fit(X_train_s, y_train)
                y_prob = model.predict_proba(X_test_s)[:, 1]

                auc = safe_auc(y_test.values, y_prob)
                pr_auc = safe_pr_auc(y_test.values, y_prob)
                brier = float(brier_score_loss(y_test.values, y_prob))
                ece = expected_calibration_error(y_test.values, y_prob)
                lift10, precision10, base_rate_fold = lift_at_k(y_test.values, y_prob, k=0.10)

                thr = best_threshold_by_utility(
                    y_true=y_test.values,
                    y_prob=y_prob,
                    tp_reward=LABEL_CFG["tp_atr_mult"],
                    sl_cost=LABEL_CFG["sl_atr_mult"],
                    min_signal_rate=0.02,
                    max_signal_rate=0.20
                )

                utility, n_signals, signal_precision = utility_score_from_probs(
                    y_true=y_test.values,
                    y_prob=y_prob,
                    threshold=thr,
                    tp_reward=LABEL_CFG["tp_atr_mult"],
                    sl_cost=LABEL_CFG["sl_atr_mult"]
                )

                fold_rows.append({
                    "fold": fold,
                    "auc": auc,
                    "pr_auc": pr_auc,
                    "brier": brier,
                    "ece": ece,
                    "lift10": lift10,
                    "precision10": precision10,
                    "threshold": thr,
                    "utility": utility,
                    "signals": n_signals,
                    "signal_precision": signal_precision,
                    "base_rate_fold": base_rate_fold
                })
                fold_thresholds.append(thr)

            if len(fold_rows) == 0:
                continue

            dfm = pd.DataFrame(fold_rows)

            row = {
                "RankKey": i,
                "Samples": int(len(X)),
                "AUC (Mean)": float(dfm["auc"].mean()),
                "AUC (Std)": float(dfm["auc"].std()) if len(dfm) > 1 else 0.0,
                "AUC Stability": float(dfm["auc"].mean() - dfm["auc"].std()) if len(dfm) > 1 else float(dfm["auc"].mean()),
                "PR_AUC (Mean)": float(dfm["pr_auc"].mean()),
                "Lift@10% (Mean)": float(dfm["lift10"].mean()),
                "Precision@10% (Mean)": float(dfm["precision10"].mean()),
                "ECE (Mean)": float(dfm["ece"].mean()),
                "Brier (Mean)": float(dfm["brier"].mean()),
                "Utility (Mean)": float(dfm["utility"].mean()),
                "AvgSignals/Fold": float(dfm["signals"].mean()),
                "SignalPrecision (Mean)": float(dfm["signal_precision"].mean()),
                "Thr": float(np.mean(fold_thresholds)) if len(fold_thresholds) > 0 else 0.70,
                "BaseRate": float(y.mean()),
                "EdgeThr": float((np.mean(fold_thresholds) if len(fold_thresholds) > 0 else 0.70) - y.mean()),
                "n_estimators": params["n_estimators"],
                "learning_rate": params["learning_rate"],
                "num_leaves": params["num_leaves"],
                "min_child_samples": params["min_child_samples"],
                "subsample": params["subsample"],
                "colsample_bytree": params["colsample_bytree"],
                "reg_alpha": params["reg_alpha"],
                "reg_lambda": params["reg_lambda"]
            }

            promoted, reasons = paper_candidate_decision(pd.Series(row))
            row["PaperCandidate"] = promoted
            row["RejectReasons"] = reasons

            # ranking compuesto
            row["CompositeScore"] = (
                row["AUC Stability"] * 3.0 +
                row["Lift@10% (Mean)"] * 1.5 +
                row["SignalPrecision (Mean)"] * 2.0 +
                row["Utility (Mean)"] * 1.5 -
                row["Brier (Mean)"] * 1.0
            )

            self.results.append(row)

            tag = "PAPER" if promoted else "REJECT"
            print(
                f"[{tag}] {i}/{len(combos)} "
                f"AUC={row['AUC (Mean)']:.4f}±{row['AUC (Std)']:.4f} | "
                f"Lift10={row['Lift@10% (Mean)']:.4f} | "
                f"SigPrec={row['SignalPrecision (Mean)']:.4f} | "
                f"Util={row['Utility (Mean)']:.4f} | "
                f"Thr={row['Thr']:.3f}"
            )

        self.finalize(X, y)

    def finalize(self, X, y):
        if len(self.results) == 0:
            print("No hubo resultados.")
            return

        results_df = pd.DataFrame(self.results).sort_values(
            by=["PaperCandidate", "CompositeScore", "AUC Stability", "Lift@10% (Mean)"],
            ascending=[False, False, False, False]
        ).reset_index(drop=True)

        print("\n" + "=" * 160)
        print("RESULTADOS V12")
        print("=" * 160)
        try:
            display(results_df)
        except Exception:
            print(results_df.to_string(index=False))

        top10 = results_df.head(10).copy()
        print("\n" + "=" * 160)
        print("TOP 10")
        print("=" * 160)
        try:
            display(top10)
        except Exception:
            print(top10.to_string(index=False))

        paper_df = results_df[results_df["PaperCandidate"] == True].copy()
        print("\n" + "=" * 160)
        print("PAPER CANDIDATES")
        print("=" * 160)
        if len(paper_df) > 0:
            try:
                display(paper_df)
            except Exception:
                print(paper_df.to_string(index=False))
        else:
            print("No hubo paper candidates con los gates actuales.")

        best = results_df.iloc[0].to_dict()
        best_params = {
            "n_estimators": int(best["n_estimators"]),
            "learning_rate": float(best["learning_rate"]),
            "num_leaves": int(best["num_leaves"]),
            "min_child_samples": int(best["min_child_samples"]),
            "subsample": float(best["subsample"]),
            "colsample_bytree": float(best["colsample_bytree"]),
            "reg_alpha": float(best["reg_alpha"]),
            "reg_lambda": float(best["reg_lambda"])
        }

        scaler = StandardScaler().fit(X)
        Xs = scaler.transform(X)
        model = build_lgbm(best_params)
        model.fit(Xs, y)

        date_str = datetime.utcnow().strftime("%Y%m%d")
        model_key = f"model_QQQ_Put_bear_volatile_B7_lgbm_focus_v12_{date_str}.pkl"
        manifest_key = f"_manifest_QQQ_Put_bear_volatile_B7_v12_{date_str}.json"
        csv_key = f"research_v12_QQQ_Put_bear_volatile_B7_{date_str}.csv"

        pack = {
            "model": model,
            "scaler": scaler
        }
        self.qb.ObjectStore.SaveBytes(model_key, pickle.dumps(pack))

        manifest = {
            "ticker": "QQQ",
            "direction": "Put",
            "regime": "bear_volatile",
            "label": LABEL_CFG,
            "model_type": "lgbm",
            "calibrated": False,
            "best_model_key": model_key,
            "features": list(X.columns),
            "threshold": float(best["Thr"]),
            "edge_threshold": float(best["EdgeThr"]),
            "base_rate": float(best["BaseRate"]),
            "paper_candidate": bool(best["PaperCandidate"]),
            "metrics": {
                "samples": int(best["Samples"]),
                "auc_mean": float(best["AUC (Mean)"]),
                "auc_std": float(best["AUC (Std)"]),
                "auc_stability": float(best["AUC Stability"]),
                "pr_auc_mean": float(best["PR_AUC (Mean)"]),
                "lift10_mean": float(best["Lift@10% (Mean)"]),
                "precision10_mean": float(best["Precision@10% (Mean)"]),
                "ece_mean": float(best["ECE (Mean)"]),
                "brier_mean": float(best["Brier (Mean)"]),
                "utility_mean": float(best["Utility (Mean)"]),
                "signal_precision_mean": float(best["SignalPrecision (Mean)"]),
                "avg_signals_fold": float(best["AvgSignals/Fold"])
            },
            "best_params": best_params,
            "reject_reasons_if_any": best["RejectReasons"]
        }

        self.qb.ObjectStore.Save(manifest_key, json.dumps(manifest, indent=4))
        self.qb.ObjectStore.Save(csv_key, results_df.to_csv(index=False))

        print("\nGuardado en ObjectStore:")
        print(f"- {model_key}")
        print(f"- {manifest_key}")
        print(f"- {csv_key}")

        print("\nMEJOR CONFIGURACIÓN:")
        for k, v in manifest.items():
            if k not in ["features", "metrics", "best_params", "label"]:
                print(f"{k}: {v}")
        print("best_params:", best_params)
        print("metrics:", manifest["metrics"])


if HAS_DEPS:
    runner = FocusedResearchV12()
    runner.run()
else:
    print("Instala dependencias requeridas: pandas_ta, scikit-learn, lightgbm.")